In [1]:
import atom_numbering as an
import pandas as pd
import get_xyz_descriptors_original as xyz
import os

In [2]:
#Provide path to directory or directory of directies containing xyz files
path = "/home/aconner/orca_descriptors/Lig411"

#Compile dataframe with atom numberings; Can specify 0 or 1 indexing for atom numberings and whether to search subdirectories or main diertory only defaults to 1 indexing and searching subdirectories
df = an.compile_final(path) 
df

added slash to end of path, new path is /home/aconner/orca_descriptors/Lig411/


,xyz_name,path,ligand_type,inchi,smiles,H1,Ni,H2,N1,C1,C2,N2,ligand_class
0,optimization,/home/aconner/orca_descriptors/Lig411/conf_2/,core_5,MTLQCWDRIPKWDR-UHFFFAOYSA-N,[H]C1=C([H])C(=C2NC(C([H])([H])[H])=C([H])C([H...,28,27,29,9,8,7,1,bpy
1,optimization,/home/aconner/orca_descriptors/Lig411/conf_1/,core_5,MTLQCWDRIPKWDR-UHFFFAOYSA-N,[H]C1=C([H])C(=C2NC(C([H])([H])[H])=C([H])C([H...,28,27,29,1,7,8,9,bpy


In [3]:
#Define lists for decriptor retrieval 
atoms = ["N1", "N2", "Ni"]
bonds = [("C1", "C2"), ("Ni", "H2")]
angles = [("N1", "Ni", "N2"), ("C1", "N1", "Ni"), ("C2", "N2", "Ni")]
dihedrals = [("C1", "N1", "Ni", "N2"), ("C2", "N2", "Ni", "N1")]
quadrants = [[2, 1, 3, 4, 6]]

In [ ]:
#generate descriptor dictionary for each row in dataframe and add to dataframe
results_dict = {}
for row in df.itertuples():

    P = xyz.PropertiesXYZ(path=df.loc[row.Index, "path"])
    
    #Zero atom properties
    P.get_SASA_and_Volume()

    #One atom properties
    atom_list = []
    for atom in atoms:
        atom_list.append(int(df.loc[row.Index, atom]))
    
    P.define_atom_list(atom_list)
    P.get_Vbur()
    P.get_atomic_SASA()

    #Two atom properties
    bond_list = []
    for bond in bonds: 
        bond_list.append([int(df.loc[row.Index, bond[0]]), int(df.loc[row.Index, bond[1]])])
    P.define_bond_list(bond_list)
    P.get_bond_distances()
    P.get_Sterimol_parameters()

    #Three atom properties
    angle_list = []
    for angle in angles:
        angle_list.append([int(df.loc[row.Index, angle[0]]), int(df.loc[row.Index, angle[1]]), int(df.loc[row.Index, angle[2]])])
    P.define_angle_list(angle_list)
    P.get_angles()

    #Four atom properties
    dihedral_list = []
    for dihedral in dihedrals:
        dihedral_list.append([int(df.loc[row.Index, dihedral[0]]), int(df.loc[row.Index, dihedral[1]]), int(df.loc[row.Index, dihedral[2]]), int(df.loc[row.Index, dihedral[3]])])
    P.define_dihedral_list(dihedral_list)
    P.get_dihedrals()
    #P.get_pyramidalizations()

    #Quadrant properties
    quadrant_list = quadrants
    P.define_quadrant_list(quadrant_list)
    #P.get_vbur_quadrants()
    #P.get_vbur_octants()

    results_dict[row.Index] = P.results



No xyz.json file found in the specified path. Results dictionary will be empty.
   /home/aconner/orca_descriptors/Lig411/conf_2/optimization.xyz / Bmin:  1.77 / Bmax:  4.96 / L:  6.35
   /home/aconner/orca_descriptors/Lig411/conf_2/optimization.xyz / Bmin:  1.95 / Bmax:  6.68 / L:  3.60


Exception: 4 neighbors. 3 expected.

In [6]:
dict_atoms = {}
for row in df.itertuples():
    dict_atoms[row.Index] = {}
    for col in df.columns:
        if col.startswith("H1"):
            atom = col[:-1] 
            number = df.at[row.Index, col]
            dict_atoms[row.Index][col] = atom + str(number)
        elif col.startswith("H2"):
            atom = col[:-1] 
            number = df.at[row.Index, col]
            dict_atoms[row.Index][col] = atom + str(number)
        elif col.startswith("C1"):
            atom = col[:-1] 
            number = df.at[row.Index, col]
            dict_atoms[row.Index][col] = atom + str(number)
        elif col.startswith("C2"):
            atom = col[:-1] 
            number = df.at[row.Index, col]
            dict_atoms[row.Index][col] = atom + str(number)
        elif col.startswith("N1"):
            atom = col[:-1] 
            number = df.at[row.Index, col]
            dict_atoms[row.Index][col] = atom + str(number)
        elif col.startswith("N2"):
            atom = col[:-1] 
            number = df.at[row.Index, col]
            dict_atoms[row.Index][col] = atom + str(number)
        elif col.startswith("Ni"):
            atom = col  
            number = df.at[row.Index, col]
            dict_atoms[row.Index][col] = atom + str(number)



In [10]:
#alter key in results dictionary to match format of user defined lists and add to dataframe
#Have to add new descriptors to this code!!
new_results = {}

for key in results_dict.keys():
    new_results[key] = {}
    for subkey in results_dict[key].keys():
        if subkey.startswith("Bond_"):
            for bond in bonds:
                bond_key = [dict_atoms[key][bond[0]], dict_atoms[key][bond[1]]]
                if all(bk in subkey for bk in bond_key):
                    new_key = "Bond_" + bond[0] + "_" + bond[1]
                    new_results[key][new_key] = results_dict[key][subkey]
        elif subkey.startswith("Sterimol_") and subkey.endswith("_L"):
            for bond in bonds:
                bond_key = [dict_atoms[key][bond[0]], dict_atoms[key][bond[1]]]
                if all(bk in subkey for bk in bond_key):
                    new_key = "Sterimol_" + bond[0] + "_" + bond[1] +"_L"
                    new_results[key][new_key] = results_dict[key][subkey]
        elif subkey.startswith("Sterimol_") and subkey.endswith("_Bmin"):
            for bond in bonds:
                bond_key = [dict_atoms[key][bond[0]], dict_atoms[key][bond[1]]]
                if all(bk in subkey for bk in bond_key):
                    new_key = "Sterimol_" + bond[0] + "_" + bond[1] +"_Bmin"
                    new_results[key][new_key] = results_dict[key][subkey]
        elif subkey.startswith("Sterimol_") and subkey.endswith("_Bmax"):
            for bond in bonds:
                bond_key = [dict_atoms[key][bond[0]], dict_atoms[key][bond[1]]]
                if all(bk in subkey for bk in bond_key):
                    new_key = "Sterimol_" + bond[0] + "_" + bond[1] +"_Bmax"
                    new_results[key][new_key] = results_dict[key][subkey]
        elif subkey.startswith("Angle_"):
            for angle in angles:
                angle_key = [dict_atoms[key][angle[0]], dict_atoms[key][angle[1]], dict_atoms[key][angle[2]]]
                if all(ak in subkey for ak in angle_key):
                    new_key = "Angle_" + angle[0] + "_" + angle[1] + "_" + angle[2]
                    new_results[key][new_key] = results_dict[key][subkey]
        elif subkey.startswith("Dihed_"):
            for dihedral in dihedrals:
                dihedral_key = [dict_atoms[key][dihedral[0]], dict_atoms[key][dihedral[1]], dict_atoms[key][dihedral[2]], dict_atoms[key][dihedral[3]]]
                if all(dk in subkey for dk in dihedral_key):
                    new_key = "Dihed_" + dihedral[0] + "_" + dihedral[1] + "_" + dihedral[2] + "_" + dihedral[3]
                    new_results[key][new_key] = results_dict[key][subkey]
        elif subkey.startswith("%Vbur_atom_"):
            for atom in atoms:
                atom_key = dict_atoms[key][atom]
                if atom_key in subkey:
                    new_key = "%Vbur_" + atom
                    new_results[key][new_key] = results_dict[key][subkey]
        elif subkey.startswith("SASA_"):
            for atom in atoms:
                atom_key = dict_atoms[key][atom]
                if atom_key in subkey:
                    new_key = "SASA_" + atom
                    new_results[key][new_key] = results_dict[key][subkey]
        else:
            new_results[key][subkey] = results_dict[key][subkey]

new_results

{0: {'SASA': 400.4419120473582,
  '%Vbur_N1': 41.59026047743685,
  '%Vbur_N2': 41.58316514774053,
  '%Vbur_Ni': 35.95065592381192,
  'SASA_N1': 1.2199870622822047,
  'SASA_N2': 1.2399868501884745,
  'SASA_Ni': 28.68899778589467,
  'Bond_C1_C2': 1.4874748257283663,
  'Bond_Ni_H2': 1.4455476119191604,
  'Sterimol_C1_C2_Bmin': 1.7725501975708888,
  'Sterimol_C1_C2_Bmax': 4.9600907249767126,
  'Sterimol_C1_C2_L': 6.349999999999999,
  'Sterimol_Ni_H2_Bmin': 1.9500000000000008,
  'Sterimol_Ni_H2_Bmax': 6.677012805139736,
  'Sterimol_Ni_H2_L': 3.6000000000000005,
  'Angle_N1_Ni_N2': 82.26112793675044,
  'Angle_C1_N1_Ni': 113.28497309812816,
  'Angle_C2_N2_Ni': 113.27729393540024,
  'Dihed_C1_N1_Ni_N2': -4.932641503772813,
  'Dihed_C2_N2_Ni_N1': 5.317129021800939},
 1: {'SASA': 400.5319182538029,
  '%Vbur_N1': 41.5827709627574,
  '%Vbur_N2': 41.58572735013087,
  '%Vbur_Ni': 35.9502617388288,
  'SASA_N1': 1.2099871683290697,
  'SASA_N2': 1.2199870622822047,
  'SASA_Ni': 28.68899778589467,
  'Bo

In [8]:
#add new results to dataframe
for key in new_results.keys():
    for subkey in new_results[key].keys():
        df.at[key, subkey] = new_results[key][subkey]

df

,xyz_name,path,ligand_type,inchi,smiles,H1,Ni,H2,N1,C1,...,%Vbur_N1,%Vbur_N2,%Vbur_Ni,Bond_N1_N2,Bond_Ni_N1,Angle_N1_Ni_N2,Angle_C1_N1_Ni,Angle_C2_N2_Ni,Dihed_C1_N1_Ni_N2,Dihed_C2_N2_Ni_N1
0,optimization,/home/aconner/orca_descriptors/Lig375/conf_2/,core_5,OGBHQHDFCDOUNF-SNVBAGLBSA-N,[H]C1=C([H])C([H])=C([H])C(=C2N[C@@]([H])(C([H...,30,29,31,1,6,...,38.842397,44.736054,34.150216,2.600746,2.015457,80.936611,114.106137,113.334761,1.883258,-2.667221
1,optimization,/home/aconner/orca_descriptors/Lig375/conf_3/,core_5,OGBHQHDFCDOUNF-SNVBAGLBSA-N,[H]C1=C([H])C([H])=C([H])C(=C2N[C@@]([H])(C([H...,30,29,31,1,6,...,38.195934,44.518464,34.611412,2.621274,2.009218,81.210911,114.041386,111.713707,-5.178545,6.424122
2,optimization,/home/aconner/orca_descriptors/Lig375/conf_4/,core_5,OGBHQHDFCDOUNF-SNVBAGLBSA-N,[H]C1=C([H])C([H])=C([H])C(=C2N[C@@]([H])(C([H...,30,29,31,1,6,...,38.563117,44.670225,34.433044,2.601459,2.021086,80.809475,114.072391,113.440548,2.432509,-3.066490
3,optimization,/home/aconner/orca_descriptors/Lig375/conf_1/,core_5,OGBHQHDFCDOUNF-SNVBAGLBSA-N,[H]C1=C([H])C([H])=C([H])C(=C2N[C@@]([H])(C([H...,30,29,31,1,6,...,38.127937,44.504273,34.582243,2.620938,2.005094,81.239155,114.107958,111.582571,-5.564358,6.915765


In [11]:
#Find min, max, and average of descriptors across dataset
descriptor_stats = {}
for column in df.iloc[:, 13:]:
    descriptor_stats[f'{column}_min'] = df[column].min()
    descriptor_stats[f'{column}_max'] = df[column].max()
    descriptor_stats[f'{column}_mean'] = df[column].mean()

descriptor_stats_df = pd.DataFrame(descriptor_stats, index=[0])
if df["smiles"].nunique() == 1:
    descriptor_stats_df.insert(0, "SMILES", df["smiles"].iloc[0])
    descriptor_stats_df.insert(1, "INCHI", df["inchi"].iloc[0])
else:
    print("Multiple SMILES strings found in dataset. Please seperate before calculating descriptor statistics.")    

descriptor_stats_df

,SMILES,INCHI,SASA_min,SASA_max,SASA_mean,SASA_Volume_min,SASA_Volume_max,SASA_Volume_mean,%Vbur_N1_min,%Vbur_N1_max,...,Angle_C1_N1_Ni_mean,Angle_C2_N2_Ni_min,Angle_C2_N2_Ni_max,Angle_C2_N2_Ni_mean,Dihed_C1_N1_Ni_N2_min,Dihed_C1_N1_Ni_N2_max,Dihed_C1_N1_Ni_N2_mean,Dihed_C2_N2_Ni_N1_min,Dihed_C2_N2_Ni_N1_max,Dihed_C2_N2_Ni_N1_mean
0,[H]C1=C([H])C([H])=C([H])C(=C2N[C@@]([H])(C([H...,OGBHQHDFCDOUNF-SNVBAGLBSA-N,413.020235,414.030277,413.620262,647.176364,647.956567,647.740899,38.127937,38.842397,...,114.081968,111.582571,113.440548,112.517897,-5.564358,2.432509,-1.606784,-3.06649,6.915765,1.901544


Below here is scrap work!

In [72]:
#alter key in results dictionary to match format of user defined lists and add to dataframe
new_results = {}

for key in storage.keys():
    if key.startswith("Bond_"):
        for bond in bonds:
            bond_key = [dict_atoms[bond[0]], dict_atoms[bond[1]]]
            if all(bk in key for bk in bond_key):
                new_key = key.replace(bond_key[0], f"{bond[0]}").replace(bond_key[1], f"{bond[1]}")
                #print(f"Original key: {key}, New key: {new_key}")
                new_storage[new_key] = storage[key]
                
    elif key.startswith("Angle_"):
        for angle in angles:
            angle_key = [dict_atoms[angle[0]], dict_atoms[angle[1]], dict_atoms[angle[2]]]
            if all(ak in key for ak in angle_key):
                new_key = key.replace(angle_key[0], f"{angle[0]}").replace(angle_key[1], f"{angle[1]}").replace(angle_key[2], f"{angle[2]}")
                #print(f"Original key: {key}, New key: {new_key}")
                new_storage[new_key] = storage[key]
    
    elif key.startswith("Dihed_"):
        for dihedral in dihedrals:
            dihedral_key = [dict_atoms[dihedral[0]], dict_atoms[dihedral[1]], dict_atoms[dihedral[2]], dict_atoms[dihedral[3]]]
            if all(dk in key for dk in dihedral_key):
                new_key = key.replace(dihedral_key[0], f"{dihedral[0]}").replace(dihedral_key[1], f"{dihedral[1]}").replace(dihedral_key[2], f"{dihedral[2]}").replace(dihedral_key[3], f"{dihedral[3]}")
                #print(f"Original key: {key}, New key: {new_key}")
                new_storage[new_key] = storage[key]
    
    elif key.startswith("%Vbur_atom_"):
        for atom in atoms:
            atom_key = dict_atoms[atom]
            if atom_key in key:
                new_key = key.replace(atom_key, f"{atom}")
                #print(f"Original key: {key}, New key: {new_key}")
                new_storage[new_key] = storage[key]

    else:
        new_storage[key] = storage[key]

            
new_storage     

KeyError: 'N1'

In [7]:
#Retrieve descriptors and add to dataframe
x = None
for row in df.itertuples():
    P = xyz.PropertiesXYZ(path=df.loc[row.Index, "path"])
    bond_list = []
    key_list = []
    for bond in bonds:
        #set up columns for all 2 atom descriptors
        df[f'{bond[0]}_{bond[1]}_length'] = None
        df['Sterimol_' + f'{bond[0]}_{bond[1]}_Bmin'] = None
        df['Sterimol_' + f'{bond[0]}_{bond[1]}_Bmax'] = None
        df['Sterimol_' + f'{bond[0]}_{bond[1]}_L'] = None
        key_list.append(f'{bond[0]}_{bond[1]}_length')   
        bond_list.append([df.loc[row.Index, bond[0]], df.loc[row.Index, bond[1]]])
    P.define_bond_list(bond_list)
    P.get_bond_distances()
    count = 0
    for bond in bond_list:
        key = 'Bond_' + f'{bond[0]}_{bond[1]}'
        df.loc[row.Index, key_list[count]] = P.results[key]
        count += 1



No xyz.json file found in the specified path. Results dictionary will be empty.
No xyz.json file found in the specified path. Results dictionary will be empty.


,xyz_name,path,ligand_type,smiles,H1,Ni,H2,N1,C1,C2,...,Sterimol_Ni_C1_L,Sterimol_Ni_C2_Bmin,Sterimol_Ni_C2_Bmax,Sterimol_Ni_C2_L,Sterimol_Ni_H1_Bmin,Sterimol_Ni_H1_Bmax,Sterimol_Ni_H1_L,Sterimol_Ni_H2_Bmin,Sterimol_Ni_H2_Bmax,Sterimol_Ni_H2_L
0,optimization,/home/aconner/orca_descriptors/Lig411/conf_2/,core_5,[H]C1=C([H])C(=C2NC(C([H])([H])[H])=C([H])C([H...,28,27,29,9,8,7,...,None,None,None,None,None,None,None,None,None,None
1,optimization,/home/aconner/orca_descriptors/Lig411/conf_1/,core_5,[H]C1=C([H])C(=C2NC(C([H])([H])[H])=C([H])C([H...,28,27,29,1,7,8,...,None,None,None,None,None,None,None,None,None,None


In [ ]:
# 
# for bond in bond_list:


    # for key in P.results.keys():
    #     for key2 in dict_atoms.keys():
    #         if key2 in key:
    #             key = key.replace(key2, dict_atoms[key2], 1)
    #     print("new key b:" +  key)
    